

**Name:** Spandana Nadagouda

**Roll Number:** UNXRNSIT-1235

**Batch:** oct 2026

**Technology:** Python and NumPy

In [ ]:
from google.colab import files
uploaded = files.upload()



Saving 20440319-GroupDNA_Minor_Project_Brief.zip to 20440319-GroupDNA_Minor_Project_Brief (2).zip


In [ ]:
import zipfile
import os

zip_path = "/content/20440319-GroupDNA_Minor_Project_Brief.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    print("Files inside ZIP:")
    for name in z.namelist():
        print(name)

Files inside ZIP:
GroupDNA_Minor_Project_Brief.pdf
hostel_bois.txt


In [ ]:
import zipfile
import os

zip_path = "/content/20440319-GroupDNA_Minor_Project_Brief.zip"
extract_path = "/content/GroupDNA"

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_path)

    print("Extraction completed!")
    print("Extracted files:")

    for name in os.listdir(extract_path):
        print(name)
else:
    print("ZIP file not found. Please upload it again.")

Extraction completed!
Extracted files:
GroupDNA_Minor_Project_Brief.pdf
hostel_bois.txt


In [ ]:
import os

file_path = "/content/GroupDNA/hostel_bois.txt"

if os.path.isfile(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    print("Dataset loaded successfully!")
    print("Total lines:", len(lines))

    print("\nFirst five lines:")
    for line in lines[:5]:
        print(line.rstrip())

else:
    print("Dataset not found at:", file_path)

Dataset loaded successfully!
Total lines: 3178

First five lines:
01/04/24, 01:12 - Messages and calls are end-to-end encrypted. No one outside of this chat, not even WhatsApp, can read or listen to them. Tap to learn more.
01/04/24, 01:13 - Priya created group "Hostel Bois 4ever"
01/04/24, 01:14 - Priya added you
01/04/24, 01:17 - Rahul: scene fix
01/04/24, 01:17 - Rahul: haan


In [ ]:
import numpy as np
from datetime import datetime, timedelta

print("Required libraries imported successfully!")

Required libraries imported successfully!


## Feature 1: Chat Parser

This feature extracts timestamps, senders and messages from the
WhatsApp chat export. It also tracks system messages, deleted
messages and media-omitted entries.

In [ ]:
# AI-assisted: Used AI to understand and develop the parsing logic.

def parse_chat(lines):
    messages = []
    participants = set()

    system_count = 0
    media_count = 0
    deleted_count = 0

    current_message = None

    for raw_line in lines:
        line = raw_line.rstrip("\r\n")

        if not line.strip():
            continue

        # Detect a new message beginning with a date.
        is_new_message = (
            len(line) >= 8
            and line[:2].isdigit()
            and line[2] in "/.-"
            and line[3:5].isdigit()
            and line[5] in "/.-"
            and line[6:8].isdigit()
        )

        if not is_new_message:
            # Append continuation lines to the previous message.
            if current_message is not None:
                current_message["text"] += "\n" + line
            continue

        parts = line.split(" - ", 1)

        if len(parts) != 2:
            system_count += 1
            current_message = None
            continue

        timestamp_text = parts[0].strip()
        content = parts[1]

        sender_parts = content.split(": ", 1)

        if len(sender_parts) != 2:
            system_count += 1
            current_message = None
            continue

        sender = sender_parts[0].strip()
        text = sender_parts[1].strip()

        participants.add(sender)

        if text == "<Media omitted>":
            media_count += 1
            current_message = None
            continue

        if text == "This message was deleted":
            deleted_count += 1
            current_message = None
            continue

        current_message = {
            "timestamp": timestamp_text,
            "sender": sender,
            "text": text
        }

        messages.append(current_message)

    return messages, participants, system_count, media_count, deleted_count


messages, participants, system_count, media_count, deleted_count = parse_chat(lines)

print("PARSER RESULTS")
print("=" * 40)
print("Parsed text messages:", len(messages))
print("Participants:", len(participants))
print("System messages:", system_count)
print("Media-omitted entries:", media_count)
print("Deleted messages:", deleted_count)

print("\nFirst five parsed messages:")

for message in messages[:5]:
    print(message)

PARSER RESULTS
Parsed text messages: 3127
Participants: 6
System messages: 4
Media-omitted entries: 32
Deleted messages: 15

First five parsed messages:
{'timestamp': '01/04/24, 01:17', 'sender': 'Rahul', 'text': 'scene fix'}
{'timestamp': '01/04/24, 01:17', 'sender': 'Rahul', 'text': 'haan'}
{'timestamp': '01/04/24, 01:18', 'sender': 'Rahul', 'text': 'kya scene'}
{'timestamp': '01/04/24, 02:13', 'sender': 'Rahul', 'text': 'abhi free hai?'}
{'timestamp': '01/04/24, 02:13', 'sender': 'Rahul', 'text': 'abey'}


## Feature 2: Group Overview

Calculate message totals, participants, date range and message
counts for each participant.

In [ ]:
message_counts = {person: 0 for person in participants}
word_counts = {person: 0 for person in participants}
timestamps = []

for message in messages:
    person = message["sender"]

    message_counts[person] += 1
    word_counts[person] += len(message["text"].split())

    timestamp_text = message["timestamp"]

    try:
        timestamp = datetime.strptime(
            timestamp_text,
            "%d/%m/%y, %H:%M"
        )
    except ValueError:
        timestamp = None

    if timestamp is not None:
        timestamps.append(timestamp)

print("GROUP OVERVIEW")
print("=" * 40)
print("Total parsed text messages:", len(messages))
print("Participants:", len(participants))

if timestamps:
    first_date = min(timestamps)
    last_date = max(timestamps)

    duration = (last_date.date() - first_date.date()).days + 1

    print("First message:", first_date)
    print("Last message:", last_date)
    print("Duration:", duration, "days")

print("\nMESSAGES PER PERSON")

sorted_people = sorted(
    message_counts,
    key=lambda person: message_counts[person],
    reverse=True
)

for person in sorted_people:
    count = message_counts[person]

    percentage = (
        count / len(messages) * 100
        if messages else 0
    )

    print(f"{person}: {count} messages ({percentage:.1f}%)")

GROUP OVERVIEW
Total parsed text messages: 3127
Participants: 6
First message: 2024-04-01 01:17:00
Last message: 2024-05-30 23:31:00
Duration: 60 days

MESSAGES PER PERSON
Rahul: 940 messages (30.1%)
Priya: 712 messages (22.8%)
Neha: 624 messages (20.0%)
Aman: 484 messages (15.5%)
Karan: 345 messages (11.0%)
Vikas: 22 messages (0.7%)


## Feature 3: Most Active Day and Hour

In [ ]:
daily_counts = {}
hourly_counts = {}

for timestamp in timestamps:
    day = timestamp.strftime("%d/%m/%y")
    hour = timestamp.hour

    daily_counts[day] = daily_counts.get(day, 0) + 1
    hourly_counts[hour] = hourly_counts.get(hour, 0) + 1

if daily_counts:
    busiest_day = max(daily_counts, key=daily_counts.get)

    print(
        "Busiest day:",
        busiest_day,
        "-",
        daily_counts[busiest_day],
        "messages"
    )
else:
    print("No valid timestamps found.")

if hourly_counts:
    busiest_hour = max(hourly_counts, key=hourly_counts.get)

    print(
        f"Busiest hour: {busiest_hour:02d}:00",
        "-",
        hourly_counts[busiest_hour],
        "messages"
    )
else:
    print("No hourly data available.")

Busiest day: 04/05/24 - 74 messages
Busiest hour: 18:00 - 244 messages


## Feature 4: Activity Heatmap Using NumPy

In [ ]:
people = sorted(participants)

person_index = {
    person: index
    for index, person in enumerate(people)
}

activity = np.zeros((len(people), 24), dtype=int)

for message in messages:
    try:
        timestamp = datetime.strptime(
            message["timestamp"],
            "%d/%m/%y, %H:%M"
        )
    except ValueError:
        continue

    row = person_index[message["sender"]]
    activity[row, timestamp.hour] += 1

print("Activity matrix shape:", activity.shape)
print("\nACTIVITY HEATMAP")
print("Person     " + " ".join(f"{h:02d}" for h in range(24)))

for index, person in enumerate(people):
    row = activity[index]
    maximum = row.max()

    symbols = []

    for value in row:
        if value == 0 or maximum == 0:
            symbols.append(".")
        elif value <= maximum * 0.25:
            symbols.append("░")
        elif value <= maximum * 0.50:
            symbols.append("▒")
        else:
            symbols.append("█")

    print(f"{person:<12}" + " ".join(symbols))

Activity matrix shape: (6, 24)

ACTIVITY HEATMAP
Person     00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23
Aman        █ █ █ █ █ . . . . . . . . . ░ ░ ░ ░ ░ ░ ░ ░ . █
Karan       . . . . . . . ░ ▒ ▒ █ ▒ █ █ █ █ █ █ █ █ █ ▒ ░ ░
Neha        . . . . . ▒ ░ ░ █ █ █ ▒ █ █ ▒ ░ █ █ █ █ █ ▒ ▒ ▒
Priya       . . . . . . ░ ▒ █ █ █ █ █ █ █ ▒ ▒ █ █ █ █ ▒ ▒ ░
Rahul       ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ █ ▒ ▒ █ █ ▒ █ █ ▒ █ █ █
Vikas       . . . . . . . ▒ █ ▒ ▒ . ▒ █ . ▒ ▒ █ █ █ ▒ ▒ ▒ █


## Feature 5: Top Words

In [ ]:
stop_words = {
    "i", "is", "the", "a", "and", "or", "to",
    "of", "in", "on", "for", "it", "this", "that",
    "you", "are", "was", "we", "my", "me"
}

word_frequency = {}

for message in messages:
    text = message["text"].lower()

    for punctuation in ".,!?;:'\"()[]{}":
        text = text.replace(punctuation, " ")

    for word in text.split():
        if word and word not in stop_words:
            word_frequency[word] = word_frequency.get(word, 0) + 1

top_words = sorted(
    word_frequency,
    key=lambda word: word_frequency[word],
    reverse=True
)[:10]

print("TOP 10 WORDS")

for word in top_words:
    print(word, "-", word_frequency[word])

TOP 10 WORDS
how - 321
guys - 318
today - 292
so - 292
about - 274
hai - 268
am - 260
at - 257
s - 226
he - 220


## Feature 6: Response-Time Analysis

In [ ]:
response_gaps = {person: [] for person in people}

for index in range(1, len(messages)):
    previous = messages[index - 1]
    current = messages[index]

    if previous["sender"] == current["sender"]:
        continue

    try:
        previous_time = datetime.strptime(
            previous["timestamp"],
            "%d/%m/%y, %H:%M"
        )

        current_time = datetime.strptime(
            current["timestamp"],
            "%d/%m/%y, %H:%M"
        )
    except ValueError:
        continue

    gap_minutes = (
        current_time - previous_time
    ).total_seconds() / 60

    if gap_minutes >= 0:
        response_gaps[current["sender"]].append(gap_minutes)

print("RESPONSE-TIME ESTIMATES")

for person in people:
    gaps = response_gaps[person]

    if gaps:
        print(
            person,
            "-",
            round(sum(gaps) / len(gaps), 2),
            "minutes average gap"
        )
    else:
        print(person, "- No intervals calculated")

RESPONSE-TIME ESTIMATES
Aman - 54.91 minutes average gap
Karan - 36.83 minutes average gap
Neha - 41.3 minutes average gap
Priya - 42.57 minutes average gap
Rahul - 35.17 minutes average gap
Vikas - 34.9 minutes average gap


## Feature 6: Response-Time Analysis

In [ ]:
if timestamps:
    first_day = min(t.date() for t in timestamps)
    last_day = max(t.date() for t in timestamps)

    all_days = []
    current_day = first_day

    while current_day <= last_day:
        all_days.append(current_day)
        current_day += timedelta(days=1)

    print("LONGEST SILENT STREAKS")

    for person in people:
        active_days = set()

        for message in messages:
            if message["sender"] != person:
                continue

            try:
                timestamp = datetime.strptime(
                    message["timestamp"],
                    "%d/%m/%y, %H:%M"
                )
                active_days.add(timestamp.date())
            except ValueError:
                continue

        streak = 0
        longest_streak = 0

        for day in all_days:
            if day not in active_days:
                streak += 1
                longest_streak = max(longest_streak, streak)
            else:
                streak = 0

        print(person, "-", longest_streak, "days")
else:
    print("No valid timestamps found.")

LONGEST SILENT STREAKS
Aman - 0 days
Karan - 0 days
Neha - 0 days
Priya - 0 days
Rahul - 0 days
Vikas - 11 days


# Step 11 — Personality Archetypes

Identify each participant's primary personality archetype using
the rules and thresholds specified in the GroupDNA project brief.

The eight archetypes are:
1. The Spammer
2. The Group Mom
3. The Night Owl
4. The Storyteller
5. The Drama Queen
6. The Ghost
7. The Comedian
8. The Question Master

In [ ]:
# STEP 11: PERSONALITY ARCHETYPE DETECTION
# AI-assisted: Used AI to understand and implement the eight rules
# specified in the GroupDNA project brief.

from datetime import datetime

# --------------------------------------------------
# 1. PREPARE MESSAGES AND PARTICIPANTS
# --------------------------------------------------

# Messages must already be parsed in the previous notebook cells.
# Each message should contain timestamp, sender and text.

people = list(participants)

person_messages = {}

for person in people:
    person_messages[person] = []

for message in messages:
    sender = message["sender"]

    if sender in person_messages:
        person_messages[sender].append(message)


# Convert supported timestamp strings to datetime objects.
# If timestamps are already datetime objects, keep them unchanged.

def parse_timestamp(value):
    if isinstance(value, datetime):
        return value

    formats = [
        "%d/%m/%y, %H:%M",
        "%d/%m/%Y, %H:%M",
        "%d/%m/%y %H:%M",
        "%d/%m/%Y %H:%M",
        "%d/%m/%y, %I:%M %p",
        "%d/%m/%Y, %I:%M %p"
    ]

    for fmt in formats:
        try:
            return datetime.strptime(value.strip(), fmt)
        except ValueError:
            pass

    return None


# Store each person's messages with valid timestamps.

dated_messages = {}

for person in people:
    dated_messages[person] = []

    for message in person_messages[person]:
        dt = parse_timestamp(message["timestamp"])

        if dt is not None:
            dated_messages[person].append(
                (dt, message["text"])
            )

# Preserve chronological order.
# Python's built-in sorted() is permitted by the brief.

for person in people:
    dated_messages[person] = sorted(
        dated_messages[person],
        key=lambda item: item[0]
    )


# --------------------------------------------------
# 2. DEFINE THE EIGHT ARCHETYPE SCORING FUNCTIONS
# --------------------------------------------------

caring_keywords = [
    "okay", "safe", "eat", "sleep", "take care",
    "are you", "please", "reminder", "drink water",
    "don't forget"
]

funny_keywords = [
    "lol", "lmao", "haha", "rofl", "lmfao"
]


# Archetype 1: THE SPAMMER
# Average consecutive messages before another person speaks.

def spammer_score(person):
    burst_lengths = []
    current_burst = 0
    previous_sender = None

    for message in messages:
        sender = message["sender"]

        if sender == person:
            if previous_sender == person:
                current_burst += 1
            else:
                current_burst = 1

        else:
            if previous_sender == person and current_burst > 0:
                burst_lengths.append(current_burst)

            current_burst = 0

        previous_sender = sender

    if previous_sender == person and current_burst > 0:
        burst_lengths.append(current_burst)

    if not burst_lengths:
        return 0

    return sum(burst_lengths) / len(burst_lengths)


# Archetype 2: THE GROUP MOM
# Count occurrences of caring keywords.

def group_mom_score(person):
    score = 0

    for message in person_messages[person]:
        text = message["text"].lower()

        for keyword in caring_keywords:
            score += text.count(keyword)

    return score


# Archetype 3: THE NIGHT OWL
# Percentage of messages sent from 23:00 through 04:59.

def night_owl_score(person):
    person_data = dated_messages[person]

    if not person_data:
        return 0

    night_messages = 0

    for dt, text in person_data:
        if dt.hour >= 23 or dt.hour < 5:
            night_messages += 1

    return night_messages / len(person_data) * 100


# Archetype 4: THE STORYTELLER
# Average words per message.

def storyteller_score(person):
    person_data = person_messages[person]

    if not person_data:
        return 0

    total_words = 0

    for message in person_data:
        total_words += len(message["text"].split())

    return total_words / len(person_data)


# Archetype 5: THE DRAMA QUEEN
# Percentage of messages that are entirely uppercase,
# excluding messages shorter than 3 characters,
# OR contain at least two exclamation marks.

def drama_queen_score(person):
    person_data = person_messages[person]

    if not person_data:
        return 0

    drama_count = 0

    for message in person_data:
        text = message["text"].strip()

        is_uppercase = (
            len(text) >= 3
            and text.isupper()
            and any(char.isalpha() for char in text)
        )

        has_exclamations = text.count("!") >= 2

        if is_uppercase or has_exclamations:
            drama_count += 1

    return drama_count / len(person_data) * 100


# Archetype 6: THE GHOST
# Percentage of calendar days in the chat period
# on which the participant sent no messages.

def ghost_score(person):
    all_timestamps = [
        dt
        for person_name in people
        for dt, text in dated_messages[person_name]
    ]

    if not all_timestamps:
        return 0

    first_day = min(all_timestamps).date()
    last_day = max(all_timestamps).date()

    total_days = (last_day - first_day).days + 1

    active_days = set()

    for dt, text in dated_messages[person]:
        active_days.add(dt.date())

    silent_days = total_days - len(active_days)

    return silent_days / total_days * 100


# Archetype 7: THE COMEDIAN
# Percentage of messages containing the specified humour keywords.

def comedian_score(person):
    person_data = person_messages[person]

    if not person_data:
        return 0

    funny_count = 0

    for message in person_data:
        text = message["text"].lower()

        for keyword in funny_keywords:
            funny_count += text.count(keyword)

    return funny_count / len(person_data) * 100


# Archetype 8: THE QUESTION MASTER
# Percentage of messages ending with '?'.

def question_master_score(person):
    person_data = person_messages[person]

    if not person_data:
        return 0

    question_count = 0

    for message in person_data:
        text = message["text"].strip()

        if text.endswith("?"):
            question_count += 1

    return question_count / len(person_data) * 100


# --------------------------------------------------
# 3. COLLECT ALL EIGHT SCORES
# --------------------------------------------------

archetype_functions = {
    "THE SPAMMER": spammer_score,
    "THE GROUP MOM": group_mom_score,
    "THE NIGHT OWL": night_owl_score,
    "THE STORYTELLER": storyteller_score,
    "THE DRAMA QUEEN": drama_queen_score,
    "THE GHOST": ghost_score,
    "THE COMEDIAN": comedian_score,
    "THE QUESTION MASTER": question_master_score
}

archetype_scores = {}

for person in people:
    archetype_scores[person] = {}

    for archetype, function in archetype_functions.items():
        archetype_scores[person][archetype] = function(person)


# --------------------------------------------------
# 4. NORMALIZE SCORES BEFORE EXCLUSIVE ASSIGNMENT
# --------------------------------------------------

# The eight metrics have different units.
# Normalize each metric relative to the highest value
# among participants so that different units can be compared.
#
# Tie-breaking rule: retain the first archetype in the
# dictionary order when normalized scores are equal.

maximum_scores = {}

for archetype in archetype_functions:
    maximum_scores[archetype] = max(
        archetype_scores[person][archetype]
        for person in people
    ) if people else 0


normalized_scores = {}

for person in people:
    normalized_scores[person] = {}

    for archetype in archetype_functions:
        maximum = maximum_scores[archetype]
        raw_score = archetype_scores[person][archetype]

        if maximum > 0:
            normalized = raw_score / maximum * 100
        else:
            normalized = 0

        normalized_scores[person][archetype] = normalized


# Assign exactly one primary archetype per participant.

assignments = {}
runner_up_assignments = {}

for person in people:
    ranked = sorted(
        normalized_scores[person].items(),
        key=lambda item: item[1],
        reverse=True
    )

    assignments[person] = ranked[0][0]

    if len(ranked) > 1:
        runner_up_assignments[person] = ranked[1][0]
    else:
        runner_up_assignments[person] = "NONE"


# --------------------------------------------------
# 5. PRINT RESULTS
# --------------------------------------------------

print("=" * 75)
print("                 GROUPDNA PERSONALITY ARCHETYPES")
print("=" * 75)

for person in people:
    print("\nParticipant:", person)

    print("Primary archetype:", assignments[person])
    print("Runner-up archetype:", runner_up_assignments[person])

    print("Raw scores:")

    for archetype, score in archetype_scores[person].items():
        print(f"  {archetype:<22} {score:.2f}")

print("\n" + "=" * 75)
print("PERSONALITY ASSIGNMENTS COMPLETE")
print("=" * 75)

                 GROUPDNA PERSONALITY ARCHETYPES

Participant: Aman
Primary archetype: THE NIGHT OWL
Runner-up archetype: THE SPAMMER
Raw scores:
  THE SPAMMER            2.77
  THE GROUP MOM          99.00
  THE NIGHT OWL          80.37
  THE STORYTELLER        5.02
  THE DRAMA QUEEN        0.00
  THE GHOST              0.00
  THE COMEDIAN           0.00
  THE QUESTION MASTER    6.61

Participant: Neha
Primary archetype: THE DRAMA QUEEN
Runner-up archetype: THE SPAMMER
Raw scores:
  THE SPAMMER            2.53
  THE GROUP MOM          23.00
  THE NIGHT OWL          4.81
  THE STORYTELLER        5.32
  THE DRAMA QUEEN        63.30
  THE GHOST              0.00
  THE COMEDIAN           0.00
  THE QUESTION MASTER    6.25

Participant: Karan
Primary archetype: THE STORYTELLER
Runner-up archetype: THE SPAMMER
Raw scores:
  THE SPAMMER            1.24
  THE GROUP MOM          29.00
  THE NIGHT OWL          2.03
  THE STORYTELLER        57.05
  THE DRAMA QUEEN        0.00
  THE GHOST        

In [ ]:

# STEP 12: GENERATE THE FINAL GROUPDNA REPORT

print("=" * 60)
print("             GROUPDNA FINAL REPORT")
print("=" * 60)

print("\nTotal participants:", len(people))

print("\nPERSONALITY ARCHETYPE ASSIGNMENTS")
print("-" * 60)

for person in people:
    print("Participant:", person)
    print("Primary Archetype:", assignments[person])
    print("Runner-up Archetype:", runner_up_assignments[person])
    print("-" * 40)

print("\nARCHETYPE DISTRIBUTION")
print("-" * 60)

for archetype in archetype_functions:
    count = 0

    for person in people:
        if assignments[person] == archetype:
            count += 1

    print(archetype, ":", count)

print("\nREPORT GENERATION COMPLETED")


             GROUPDNA FINAL REPORT

Total participants: 6

PERSONALITY ARCHETYPE ASSIGNMENTS
------------------------------------------------------------
Participant: Aman
Primary Archetype: THE NIGHT OWL
Runner-up Archetype: THE SPAMMER
----------------------------------------
Participant: Neha
Primary Archetype: THE DRAMA QUEEN
Runner-up Archetype: THE SPAMMER
----------------------------------------
Participant: Karan
Primary Archetype: THE STORYTELLER
Runner-up Archetype: THE SPAMMER
----------------------------------------
Participant: Priya
Primary Archetype: THE GROUP MOM
Runner-up Archetype: THE QUESTION MASTER
----------------------------------------
Participant: Vikas
Primary Archetype: THE GHOST
Runner-up Archetype: THE COMEDIAN
----------------------------------------
Participant: Rahul
Primary Archetype: THE SPAMMER
Runner-up Archetype: THE COMEDIAN
----------------------------------------

ARCHETYPE DISTRIBUTION
------------------------------------------------------------

In [ ]:

# STEP 13.1: CHECK PARTICIPANTS

print("Total participants:", len(people))

for person in people:
    print(person)


Total participants: 6
Aman
Neha
Karan
Priya
Vikas
Rahul


In [ ]:

# STEP 13.2: CHECK ARCHETYPE ASSIGNMENTS

for person in people:
    print(person, "->", assignments[person])


Aman -> THE NIGHT OWL
Neha -> THE DRAMA QUEEN
Karan -> THE STORYTELLER
Priya -> THE GROUP MOM
Vikas -> THE GHOST
Rahul -> THE SPAMMER


In [ ]:

# STEP 13.3: CHECK ARCHETYPE COUNTS

for archetype in archetype_functions:
    count = 0

    for person in people:
        if assignments[person] == archetype:
            count += 1

    print(archetype, ":", count)


THE SPAMMER : 1
THE GROUP MOM : 1
THE NIGHT OWL : 1
THE STORYTELLER : 1
THE DRAMA QUEEN : 1
THE GHOST : 1
THE COMEDIAN : 0
THE QUESTION MASTER : 0
